In [94]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## ASSIGNMENTS - ARGUMENT MINING


This notebook uses the token classification API from 🤗 Transformers to fine-tune a model for Argument Component Detection — identifying spans of text as Claims, Premises, or non-argumentative (O). All the code required for the fine-tuning is already available in this notebook. The ASSIGNMENTS for this lab will consist of adapting the code to fine-tune models for Argument Mining.

In order to do that, we will be using the ABSTRCT corpus, a dataset of clinical trials annotated with argumentative structure, including argument components and their relations:

````
https://gitlab.com/santimarro/abstrct/-/tree/master/
````

For this notebook we provide their data already formatted for Sequence Labelling (Argument Mining or Argument Component detection) and for Argument Relation classification. You can find the datasets in this folder:

````
/content/drive/My Drive/Colab Notebooks/2025-ILTAPP/datasets/argumentation/
````

The Argument Mining assignments consists of the following steps:

+ Loading your own local datasets for Argument Mining (available in the folder "/content/drive/My Drive/Colab Notebooks/2025-ILTAPP/datasets/argumentation/"
++ HINT: look at the local argumentation data and check how you need to adapt or use the dataset loading script from conll2003 already available in the Hub (https://huggingface.co/docs/datasets/v1.2.1/add_dataset html#testing-the-dataset-loading-script)
+ Finetune the model for argument mining/detection.
+ Use the model to perform argument mining.
++ HINT: Extract some sentences from the argumentation dev/test data to make sure that some of them contain arguments from the same domain as the training data.


In [95]:
! pip install datasets transformers seqeval accelerate

In [96]:
import transformers

print(transformers.__version__)

5.0.0


Check GPU

In [97]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count()
if n_gpu > 0:
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    print('Using CPU')


Using GPU: Tesla T4



# Fine-tuning a model for Argument Mining

This notebook uses the token classification API from 🤗 Transformers to fine-tune a model for Argument Component Detection — identifying spans of text as Claims, Premises, or non-argumentative (O).

This notebook will work with any model checkpoint from the [Model Hub](https://huggingface.co/models) as long as that model has a version with a token classification head and a fast tokenizer (check on [this table](https://huggingface.co/transformers/index.html#bigtable) if this is the case). It might just need some small adjustments if you decide to use a different dataset than the one used here. Depending on you model and the GPU you are using, you might need to adjust the batch size to avoid out-of-memory errors. Set those three parameters, then the rest of the notebook should run smoothly:

In [98]:
task = "ner" # Should be one of "ner", "pos" or "chunk"
model_checkpoint = "distilbert-base-uncased"
batch_size = 16

## Loading the dataset

We will use the [🤗 Datasets](https://github.com/huggingface/datasets) library to download the data and get the metric we need to use for evaluation (to compare our model to the benchmark). This can be easily done with the functions `load_dataset` and `load_metric`.  

In [99]:
from datasets import load_dataset

## TODO Loading local argument mining dataset

For our example here, we'll use the [CONLL 2003 dataset](https://huggingface.co/datasets/eriktks/conll2003). If you're using your own dataset defined from a JSON or csv file (see the [Datasets documentation](https://huggingface.co/docs/datasets/loading_datasets.html#from-local-files) on how to load them).

+ TODO: modify the load_dataset code below to load your local argumentation files using the dataset loading script which you have adapted for that purpose from the [conll2003 script](https://huggingface.co/datasets/eriktks/conll2003/blob/main/conll2003.py).
++ HINT: look at the local argumentation data and check how you need to adapt or use the dataset loading script from conll2003 already available in the Hub (https://huggingface.co/docs/datasets/v1.2.1/add_dataset.html#testing-the-dataset-loading-script)

In [100]:
from datasets import Dataset, DatasetDict, Features, Sequence, ClassLabel, Value
import os

# Define file paths for the local argumentation datasets
data_dir = "/content/drive/My Drive/Colab Notebooks/2026-ILTAPP/2026-ILTAPP/datasets/argumentation/"
train_file = os.path.join(data_dir, "train_mining.tsv")
dev_file = os.path.join(data_dir, "dev_mining.tsv")
test_file = os.path.join(data_dir, "test_mining.tsv")

def read_conll_file(filepath):
    """Reads a CoNLL-like file and parses it into tokens and tags."""
    data = []
    current_tokens = []
    current_tags = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                # Assuming tab-separated format: token\ttag
                parts = line.split('\t')
                if len(parts) == 2:
                    current_tokens.append(parts[0])
                    current_tags.append(parts[1])
                # You might need to add error handling for malformed lines
            else:
                # Empty line indicates end of a sentence
                if current_tokens:
                    data.append({"tokens": current_tokens, "tags": current_tags})
                current_tokens = []
                current_tags = []
    # Add the last sentence if the file does not end with an empty line
    if current_tokens:
        data.append({"tokens": current_tokens, "tags": current_tags})
    return data

# Load raw data from the TSV files
train_data_raw = read_conll_file(train_file)
dev_data_raw = read_conll_file(dev_file)
test_data_raw = read_conll_file(test_file)

# Collect all unique tags to create a label map for the dataset
all_tags = set()
for sample in train_data_raw:
    all_tags.update(sample["tags"])
# Sort tags to ensure consistent ID mapping across runs
label_list = sorted(list(all_tags))
tag_to_id = {tag: i for i, tag in enumerate(label_list)}

# Convert string tags to integer IDs
def convert_tags_to_ids(data_raw, tag_to_id_map):
    processed_data = []
    for sample in data_raw:
        tag_ids = [tag_to_id_map[tag] for tag in sample["tags"]]
        processed_data.append({"tokens": sample["tokens"], f"{task}_tags": tag_ids})
    return processed_data

# Process raw data to include integer tag IDs
train_data_processed = convert_tags_to_ids(train_data_raw, tag_to_id)
dev_data_processed = convert_tags_to_ids(dev_data_raw, tag_to_id)
test_data_processed = convert_tags_to_ids(test_data_raw, tag_to_id)

# Define features for the dataset (needed for ClassLabel to provide label names)
features = Features({
    "tokens": Sequence(feature=Value(dtype='string')),
    f"{task}_tags": Sequence(feature=ClassLabel(names=label_list)),
})

# Create Dataset objects from the processed lists
train_dataset = Dataset.from_list(train_data_processed, features=features)
dev_dataset = Dataset.from_list(dev_data_processed, features=features)
test_dataset = Dataset.from_list(test_data_processed, features=features)

# Combine into a DatasetDict
datasets = DatasetDict({
    "train": train_dataset,
    "validation": dev_dataset,
    "test": test_dataset
})

# Re-assign label_list from the newly created dataset features for consistency with downstream cells
label_list = datasets["train"].features[f"{task}_tags"].feature.names

The `datasets` object itself is [`DatasetDict`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.DatasetDict), which contains one key for the training, validation and test set.

In [101]:
datasets

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 4418
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 679
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1258
    })
})

We can see the training, validation and test sets all have a column for the tokens (the input texts split into words) and one column of labels for each kind of task we introduced before.

To access an actual element, you need to select a split first, then give an index:

In [102]:
datasets["train"][0]

{'tokens': ['Facial',
  'hirsutism',
  'is',
  'one',
  'of',
  'the',
  'characteristic',
  'features',
  'of',
  'polycystic',
  'ovary',
  'syndrome',
  '(',
  'PCOS',
  ')',
  ',',
  'and',
  'this',
  'can',
  'lead',
  'to',
  'high',
  'levels',
  'of',
  'depression',
  'and',
  'anxiety',
  '.'],
 'ner_tags': [0,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2]}

The labels are already coded as integer ids to be easily usable by our model, but the correspondence with the actual categories is stored in the `features` of the dataset:

In [103]:
datasets["train"].features[f"ner_tags"]

List(ClassLabel(names=['B-Claim', 'B-Premise', 'I-Claim', 'I-Premise', 'O']))

The labels use BIO encoding over two argument component types:
- `O` — non-argumentative token
- `B-Claim` / `I-Claim` — beginning/inside a Claim span
- `B-Premise` / `I-Premise` — beginning/inside a Premise span

Since the labels are lists of `ClassLabel`, the actual names of the labels are nested in the `feature` attribute of the object above:

In [104]:
label_list = datasets["train"].features[f"{task}_tags"].feature.names
label_list

['B-Claim', 'B-Premise', 'I-Claim', 'I-Premise', 'O']

To get a sense of what the data looks like, the following function will show some examples picked randomly in the dataset (automatically decoding the labels in passing).

In [105]:
from datasets import ClassLabel, Sequence
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
        elif isinstance(typ, Sequence) and isinstance(typ.feature, ClassLabel):
            df[column] = df[column].transform(lambda x: [typ.feature.names[i] for i in x])
    display(HTML(df.to_html()))

In [106]:
show_random_elements(datasets["train"])

,tokens,ner_tags
0,"[Further, investigation, in, larger, numbers, of, patients, is, warranted, .]","[O, O, O, O, O, O, O, O, O, O]"
1,"[Five, years, later, ,, these, 190, men, underwent, HRQOL, evaluation, by, using, the, cancer-specific, 50-item, Expanded, Prostate, Cancer, Index, Composite, ,, the, Short, Form, 12, Physical, Component, Score, ,, and, Short, Form, 12, Mental, Component, Score, .]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]"
2,"[There, were, no, significant, differences, in, either, PFS, or, OS, between, IFN1, and, IFN5, (, median, of, 3.7, months, and, median, of, 3.4, months, PFS, ,, respectively, ;, median, of, 25.5, months, and, median, of, 17.5, months, OS, ,, respectively, ), .]","[B-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise]"
3,"[QOL, decreased, during, the, first, 8, weeks, of, treatment, ,, and, a, partial, recovery, followed, .]","[B-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise]"
4,"[The, primary, end, point, was, progression-free, survival, .]","[O, O, O, O, O, O, O, O]"
5,"[A, regression, analysis, with, interactions, was, performed, to, determine, if, domains, of, quality, of, life, (, EORTC-QLQ-C30, ), functioning, (, Health, Survey, Short, Form-36, ), or, psychological, distress, (, Symptom, Checklist-90, ), moderated, the, effect, of, CBT, on, fatigue, .]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]"
6,"[Grade, 2-3, alopecia, (, 68, %, PE, vs, 17, %, GC, ), and, nausea, (, 43, %, PE, vs, 26, %, GC, ), were, more, frequent, with, PE, .]","[B-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise, I-Premise]"
7,"[Patients, with, stage, III, or, IV, cancer, (, N, =, 120, ), were, randomly, assigned, to, seven, sessions, of, either, IMCP, or, therapeutic, massage, (, TM, ), .]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]"
8,"[To, compare, the, efficacy, of, a, standard, anthracycline-based, regimen, to, a, dose-intensified, anthracycline, regimen, in, locally, advanced, breast, cancer, .]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]"
9,"[To, test, the, ability, of, the, cytoprotectant, ,, amifostine, ,, to, reduce, chemoradiotherapy-induced, esophagitis, and, evaluate, its, influence, on, quality, of, life, (, QOL, ), and, swallowing, symptoms, .]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]"


## Preprocessing the data

Before we can feed those texts to our model, we need to preprocess them. This is done by a 🤗 Transformers `Tokenizer` which will (as the name indicates) tokenize the inputs (including converting the tokens to their corresponding IDs in the pretrained vocabulary) and put it in a format the model expects, as well as generate the other inputs that model requires.

To do all of this, we instantiate our tokenizer with the `AutoTokenizer.from_pretrained` method, which will ensure:

- we get a tokenizer that corresponds to the model architecture we want to use,
- we download the vocabulary used when pretraining this specific checkpoint.

That vocabulary will be cached, so it's not downloaded again the next time we run the cell.

In [107]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

The following assertion ensures that our tokenizer is a fast tokenizers (backed by Rust) from the 🤗 Tokenizers library. Those fast tokenizers are available for almost all models, and we will need some of the special features they have for our preprocessing.

In [108]:
import transformers
assert isinstance(tokenizer, transformers.PreTrainedTokenizerFast)

You can check which type of models have a fast tokenizer available and which don't on the [big table of models](https://huggingface.co/transformers/index.html#bigtable).

You can directly call this tokenizer on one sentence:

In [109]:
tokenizer("Hello, this is one sentence!")

{'input_ids': [101, 7592, 1010, 2023, 2003, 2028, 6251, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

Depending on the model you selected, you will see different keys in the dictionary returned by the cell above. They don't matter much for what we're doing here (just know they are required by the model we will instantiate later), you can learn more about them in [this tutorial](https://huggingface.co/transformers/preprocessing.html) if you're interested.

If, as is the case here, your inputs have already been split into words, you should pass the list of words to your tokenzier with the argument `is_split_into_words=True`:

In [110]:
tokenizer(["Hello", ",", "this", "is", "one", "sentence", "split", "into", "words", "."], is_split_into_words=True)

{'input_ids': [101, 7592, 1010, 2023, 2003, 2028, 6251, 3975, 2046, 2616, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Note that transformers are often pretrained with subword tokenizers, meaning that even if your inputs have been split into words already, each of those words could be split again by the tokenizer. Let's look at an example of that:

In [111]:
example = datasets["train"][4]
print(example["tokens"])

['The', 'main', 'outcomes', 'were', 'self-reported', 'severity', 'of', 'facial', 'hair', '(', 'measured', 'on', 'a', 'scale', 'of', '1-10', ')', ',', 'depression', ',', 'anxiety', '(', 'measured', 'on', 'the', 'Hospital', 'Anxiety', 'and', 'Depression', 'Scale', ')', 'and', 'quality', 'of', 'life', '(', 'measured', 'on', 'the', 'WHOQOL-BREF', ')', '.']


In [112]:
tokenized_input = tokenizer(example["tokens"], is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
print(tokens)

['[CLS]', 'the', 'main', 'outcomes', 'were', 'self', '-', 'reported', 'severity', 'of', 'facial', 'hair', '(', 'measured', 'on', 'a', 'scale', 'of', '1', '-', '10', ')', ',', 'depression', ',', 'anxiety', '(', 'measured', 'on', 'the', 'hospital', 'anxiety', 'and', 'depression', 'scale', ')', 'and', 'quality', 'of', 'life', '(', 'measured', 'on', 'the', 'who', '##q', '##ol', '-', 'br', '##ef', ')', '.', '[SEP]']


Some clinical terms and compound words in the ABSTRCT corpus will be split into subtokens by DistilBERT's wordpiece tokenizer.

This means that we need to do some processing on our labels as the input ids returned by the tokenizer are longer than the lists of labels our dataset contain, first because some special tokens might be added (we can a `[CLS]` and a `[SEP]` above) and then because of those possible splits of words in multiple tokens:

In [113]:
len(example[f"{task}_tags"]), len(tokenized_input["input_ids"])

(42, 53)

Thankfully, the tokenizer returns outputs that have a `word_ids` method which can help us.

In [114]:
print(tokenized_input.word_ids())

[None, 0, 1, 2, 3, 4, 4, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 15, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 39, 39, 39, 39, 39, 40, 41, None]


As we can see, it returns a list with the same number of elements as our processed input ids, mapping special tokens to `None` and all other tokens to their respective word. This way, we can align the labels with the processed input ids.

In [115]:
word_ids = tokenized_input.word_ids()
aligned_labels = [-100 if i is None else example[f"{task}_tags"][i] for i in word_ids]
print(len(aligned_labels), len(tokenized_input["input_ids"]))

53 53


Here we set the labels of all special tokens to -100 (the index that is ignored by PyTorch) and the labels of all other tokens to the label of the word they come from. Another strategy is to set the label only on the first token obtained from a given word, and give a label of -100 to the other subtokens from the same word. We propose the two strategies here, just change the value of the following flag:

In [116]:
label_all_tokens = True

We're now ready to write the function that will preprocess our samples. We feed them to the `tokenizer` with the argument `truncation=True` (to truncate texts that are bigger than the maximum size allowed by the model) and `is_split_into_words=True` (as seen above). Then we align the labels with the token ids using the strategy we picked:

In [117]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"{task}_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

This function works with one or several examples. In the case of several examples, the tokenizer will return a list of lists for each key:

In [118]:
tokenize_and_align_labels(datasets['train'][:5])

{'input_ids': [[101, 13268, 7632, 2869, 21823, 6491, 2003, 2028, 1997, 1996, 8281, 2838, 1997, 26572, 5666, 10074, 1051, 21639, 8715, 1006, 7473, 2891, 1007, 1010, 1998, 2023, 2064, 2599, 2000, 2152, 3798, 1997, 6245, 1998, 10089, 1012, 102], [101, 2000, 16157, 1996, 4254, 1997, 9138, 3949, 2006, 1996, 18976, 1997, 13268, 7632, 2869, 21823, 6491, 1998, 2006, 8317, 22822, 17062, 3012, 1999, 2308, 2007, 7473, 2891, 1012, 102], [101, 1037, 6721, 3550, 4758, 3979, 1997, 2274, 2152, 1011, 19857, 10127, 13441, 1006, 8830, 1007, 5443, 1012, 2274, 2659, 1011, 19857, 10127, 13441, 1006, 2491, 1007, 2001, 2864, 2058, 1020, 2706, 1999, 1037, 2120, 2740, 2326, 4252, 2902, 1012, 102], [101, 5739, 2020, 6070, 2308, 2007, 13268, 7632, 2869, 21823, 6491, 2349, 2000, 7473, 2891, 8733, 2013, 2902, 2041, 24343, 17865, 1998, 1037, 5776, 2490, 2177, 1999, 2541, 1011, 2526, 1012, 102], [101, 1996, 2364, 13105, 2020, 2969, 1011, 2988, 18976, 1997, 13268, 2606, 1006, 7594, 2006, 1037, 4094, 1997, 1015, 1011, 

To apply this function on all the sentences (or pairs of sentences) in our dataset, we just use the `map` method of our `dataset` object we created earlier. This will apply the function on all the elements of all the splits in `dataset`, so our training, validation and testing data will be preprocessed in one single command.

In [119]:
tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/4418 [00:00<?, ? examples/s]

Map:   0%|          | 0/679 [00:00<?, ? examples/s]

Map:   0%|          | 0/1258 [00:00<?, ? examples/s]

Even better, the results are automatically cached by the 🤗 Datasets library to avoid spending time on this step the next time you run your notebook. The 🤗 Datasets library is normally smart enough to detect when the function you pass to map has changed (and thus requires to not use the cache data). For instance, it will properly detect if you change the task in the first cell and rerun the notebook. 🤗 Datasets warns you when it uses cached files, you can pass `load_from_cache_file=False` in the call to `map` to not use the cached files and force the preprocessing to be applied again.

Note that we passed `batched=True` to encode the texts by batches together. This is to leverage the full benefit of the fast tokenizer we loaded earlier, which will use multi-threading to treat the texts in a batch concurrently.

## Fine-tuning the model

Now that our data is ready, we can download the pretrained model and fine-tune it. Since all our tasks are about token classification, we use the `AutoModelForTokenClassification` class. Like with the tokenizer, the `from_pretrained` method will download and cache the model for us. The only thing we have to specify is the number of labels for our problem (which we can get from the features, as seen before):

In [120]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


The warning is telling us we are throwing away some weights (the `vocab_transform` and `vocab_layer_norm` layers) and randomly initializing some other (the `pre_classifier` and `classifier` layers). This is absolutely normal in this case, because we are removing the head used to pretrain the model on a masked language modeling objective and replacing it with a new head for which we don't have pretrained weights, so the library warns us we should fine-tune this model before using it for inference, which is exactly what we are going to do.

To instantiate a `Trainer`, we will need to define three more things. The most important is the [`TrainingArguments`](https://huggingface.co/transformers/main_classes/trainer.html#transformers.TrainingArguments), which is a class that contains all the attributes to customize the training. It requires one folder name, which will be used to save the checkpoints of the model, and all other arguments are optional:

In [121]:
model_name = model_checkpoint.split("/")[-1]
args = TrainingArguments(
    f"/content/drive/My Drive/Colab Notebooks/2025-ILTAPP/resources/transformers-ner-model",
    eval_strategy = "epoch", # Changed from evaluation_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
)

Here we set the evaluation to be done at the end of each epoch, tweak the learning rate, use the `batch_size` defined at the top of the notebook and customize the number of epochs for training, as well as the weight decay.

Then we will need a data collator that will batch our processed examples together while applying padding to make them all the same size (each pad will be padded to the length of its longest example). There is a data collator for this task in the Transformers library, that not only pads the inputs, but also the labels:

In [122]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

The last thing to define for our `Trainer` is how to compute the metrics from the predictions. Here we will load the [`seqeval`](https://github.com/chakki-works/seqeval) metric (which is commonly used to evaluate results on the CONLL dataset) via the Datasets library.

In [123]:
!pip install evaluate
import evaluate
metric = evaluate.load("seqeval")

This metric takes list of labels for the predictions and references:

In [124]:
labels = [label_list[i] for i in example[f"{task}_tags"]]
metric.compute(predictions=[labels], references=[labels])

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:557: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


{'overall_precision': np.float64(0.0),
 'overall_recall': np.float64(0.0),
 'overall_f1': np.float64(0.0),
 'overall_accuracy': 1.0}

So we will need to do a bit of post-processing on our predictions:
- select the predicted index (with the maximum logit) for each token
- convert it to its string label
- ignore everywhere we set a label of -100

The following function does all this post-processing on the result of `Trainer.evaluate` (which is a namedtuple containing predictions and labels) before applying the metric:

In [125]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Note that we drop the precision/recall/f1 computed for each category and only focus on the overall precision/recall/f1/accuracy.

Then we just need to pass all of this along with our datasets to the `Trainer`:

In [126]:
trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

We can now finetune our model by just calling the `train` method:

In [88]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.320113,0.372233,0.463659,0.412946,0.874019
2,0.402318,0.287908,0.319927,0.438596,0.369979,0.888635
3,0.402318,0.310852,0.367906,0.471178,0.413187,0.890338


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=831, training_loss=0.3268130562939512, metrics={'train_runtime': 175.9122, 'train_samples_per_second': 75.344, 'train_steps_per_second': 4.724, 'total_flos': 303833422270020.0, 'train_loss': 0.3268130562939512, 'epoch': 3.0})

The `evaluate` method allows you to evaluate again on the evaluation dataset or on another dataset:

In [89]:
trainer.evaluate()

{'eval_loss': 0.31085240840911865,
 'eval_precision': 0.3679060665362035,
 'eval_recall': 0.47117794486215536,
 'eval_f1': 0.41318681318681316,
 'eval_accuracy': 0.8903375825270938,
 'eval_runtime': 2.8918,
 'eval_samples_per_second': 234.799,
 'eval_steps_per_second': 14.869,
 'epoch': 3.0}

To get the precision/recall/f1 computed for each category now that we have finished training, we can apply the same function as before on the result of the `predict` method:

In [90]:
predictions, labels, _ = trainer.predict(tokenized_datasets["validation"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

results = metric.compute(predictions=true_predictions, references=true_labels)
results

{'Claim': {'precision': np.float64(0.19895287958115182),
  'recall': np.float64(0.2585034013605442),
  'f1': np.float64(0.22485207100591717),
  'number': np.int64(147)},
 'Premise': {'precision': np.float64(0.46875),
  'recall': np.float64(0.5952380952380952),
  'f1': np.float64(0.5244755244755245),
  'number': np.int64(252)},
 'overall_precision': np.float64(0.3679060665362035),
 'overall_recall': np.float64(0.47117794486215536),
 'overall_f1': np.float64(0.41318681318681316),
 'overall_accuracy': 0.8903375825270938}

from transformers import pipeline

# Path to the model saved by trainer.save_model() above
saved_model_path = f"/content/drive/My Drive/Colab Notebooks/2025-ILTAPP/resources/transformers-ner-model"

# Explicitly save the model to ensure config.json is present and correct
trainer.save_model(saved_model_path)

token_classifier = pipeline(
    "token-classification", model=saved_model_path, aggregation_strategy="simple"
)

# Quick in-domain smoke test — pull a sentence from the validation set
sample_tokens = datasets["validation"][0]["tokens"]
text = " ".join(sample_tokens[:min(len(sample_tokens), 64)])
print(text)
token_classifier(text)

In [91]:
saved_model_path = "/content/drive/My Drive/Colab Notebooks/2025-ILTAPP/resources/transformers-ner-model"


# Save with correct label mappings
model.config.id2label = {i: label for i, label in enumerate(label_list)}
model.config.label2id = {label: i for i, label in enumerate(label_list)}
trainer.save_model(saved_model_path)

# Load pipeline
token_classifier = pipeline(
    "token-classification", model=saved_model_path, aggregation_strategy="simple"
)

# Smoke test on a validation sentence
sample_tokens = datasets["validation"][0]["tokens"]
text = " ".join(sample_tokens[:min(len(sample_tokens), 64)])
print(text)
token_classifier(text)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Implant-based reconstruction is performed in the majority of women offered primary reconstruction for breast cancer .


[]

In [92]:
predictions, labels, _ = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

test_results = metric.compute(predictions=true_predictions, references=true_labels)
test_results

{'Claim': {'precision': np.float64(0.22598870056497175),
  'recall': np.float64(0.20151133501259447),
  'f1': np.float64(0.21304926764314247),
  'number': np.int64(397)},
 'Premise': {'precision': np.float64(0.44095665171898357),
  'recall': np.float64(0.5286738351254481),
  'f1': np.float64(0.48084759576202124),
  'number': np.int64(558)},
 'overall_precision': np.float64(0.36656891495601174),
 'overall_recall': np.float64(0.39267015706806285),
 'overall_f1': np.float64(0.37917087967644086),
 'overall_accuracy': 0.8807362240289069}

# - ERROR ANALYSIS

+ Check the results obtained. Are they similar to the state of the art?
+ Check the predictions. What seem to be the most common errors?


In [93]:
# 1. Per-class results (not just overall)
import pprint

# Re-predict on the validation set to ensure true_predictions and true_labels match datasets["validation"]
predictions_val, labels_val, _ = trainer.predict(tokenized_datasets["validation"])
predictions_val = np.argmax(predictions_val, axis=2)

true_predictions_val = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions_val, labels_val)
]
true_labels_val = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions_val, labels_val)
]

pprint.pprint(metric.compute(predictions=true_predictions_val, references=true_labels_val))

# 2. Find sentences where the model makes errors
errors = []
for sample, preds, gold in zip(datasets["validation"], true_predictions_val, true_labels_val):
    if preds != gold:
        errors.append({
            "tokens": sample["tokens"],
            "gold": gold,
            "predicted": preds
        })

print(f"Sentences with at least one error: {len(errors)} / {len(datasets['validation'])}")
print()

# Show first 5 error cases
for e in errors[:5]:
    print("Tokens:   ", e["tokens"])
    print("Gold:     ", e["gold"])
    print("Predicted:", e["predicted"])
    print()

{'Claim': {'f1': np.float64(0.22485207100591717),
           'number': np.int64(147),
           'precision': np.float64(0.19895287958115182),
           'recall': np.float64(0.2585034013605442)},
 'Premise': {'f1': np.float64(0.5244755244755245),
             'number': np.int64(252),
             'precision': np.float64(0.46875),
             'recall': np.float64(0.5952380952380952)},
 'overall_accuracy': 0.8903375825270938,
 'overall_f1': np.float64(0.41318681318681316),
 'overall_precision': np.float64(0.3679060665362035),
 'overall_recall': np.float64(0.47117794486215536)}
Sentences with at least one error: 186 / 679

Tokens:    ['Of', 'the', 'patients', 'in', 'the', 'one-stage', 'group', ',', '70', 'percent', 'had', 'revision', 'surgery', ',', 'mostly', 'because', 'of', 'upper', 'pole', 'fullness', 'and', 'poor', 'ptosis', '.']
Gold:      ['B-Premise', 'I-Premise', 'I-Premise', 'I-Premise', 'I-Premise', 'I-Premise', 'I-Premise', 'I-Premise', 'I-Premise', 'I-Premise', 'I-Premise', 

## Error Analysis

### Results vs. state of the art

The ABSTRCT paper (Mayer et al., 2020) [link](https://ecai2020.eu/papers/1470_paper.pdf) reports a macro F1 of 0.87 for argument component detection, using bidirectional transformers combined with LSTM/GRU/CRF layers. My fine-tuned DistilBERT baseline achieves F1 0.38 on the test set. The gap reflects both the architectural difference — a plain fine-tuned model versus a purpose-built pipeline — and the domain mismatch between general-domain DistilBERT and clinical trial text.

### Per-class breakdown (test set)

| Class   | Precision | Recall | F1   | Support |
|---------|-----------|--------|------|---------|
| Claim   | 0.23      | 0.20   | 0.21 | 397     |
| Premise | 0.44      | 0.53   | 0.48 | 558     |

Premises are detected considerably more reliably than Claims. Claims tend to be short evaluative statements with varied phrasing, and the model struggles to identify them consistently — both precision and recall are very low. For Premises, recall exceeds precision, indicating the model over-generates Premise spans. The training curve also showed some instability, with validation F1 dipping at epoch 2 before partially recovering at epoch 3.

### Error patterns

Four main error types were observed in the validation output:

1. **Spurious spans** — `B-Premise` predicted on sentences that are entirely O (e.g. *"These findings agreed..."*, *"The 5-year overall survival was 47%..."*), suggesting the model associates statistical and result-reporting sentences with Premises regardless of their argumentative status.
2. **Missed span onsets** — the model correctly identifies part of a gold span but misses the opening tokens, producing fragmented coverage. Since seqeval requires a correct `B-` tag to score a span, even partially correct predictions receive no credit.
3. **Class confusion throughout spans** — rather than just at boundaries, tokens flip between Claim and Premise repeatedly within the same span (e.g. *"The permanent expander method failed..."* predicted as B-Premise I-Premise I-Claim I-Claim I-Premise...). This reflects poor separation between the two classes.
4. **Fragmented spans** — for longer Claims the model drops in and out mid-sequence, consistent with DistilBERT's reduced capacity relative to full BERT.